In [1]:
! pip install -q torch_geometric
! pip install -q cell-gears
! pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
! pip -q install scikit-misc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5

In [2]:
!pip install crc32c

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 3.7 MB/s eta 0:00:00


In [3]:

from gears import PertData
from copy import deepcopy
import os
import pickle
from torch.optim.lr_scheduler import StepLR
from torch import optim
from torch_geometric.data import Data
from torch_geometric.nn import knn_graph
from torch_geometric.loader import DataLoader
from gears.gears import GEARS
from gears.model import GEARS_Model
import anndata as ad
from gears.data_utils import DataSplitter,print_sys
import torch
import scanpy as sc
import random


In [4]:
import torch
import torch.nn.functional as F
import numpy as np

def rbf_kernel(x, y, gamma=None):
    """
    Computes the RBF kernel using torch.cdist (optimized C++ implementation).
    """
    if gamma is None:
        gamma = 1.0 / x.size(1)
    
    # p=2 is Euclidean distance
    dist_sq = torch.cdist(x, y, p=2).pow(2)
    K = torch.exp(-gamma * dist_sq)
    return K

def maximum_mean_discrepancy(x, y, gamma=None):
    """
    MMD: Measures distance between distributions.
    """
    if x.size(0) < 2 or y.size(0) < 2:
        # MMD is undefined/unstable for single-sample batches
        return torch.tensor(0.0, device=x.device)

    K_xx = rbf_kernel(x, x, gamma)
    K_yy = rbf_kernel(y, y, gamma)
    K_xy = rbf_kernel(x, y, gamma)

    mmd_val = K_xx.mean() + K_yy.mean() - 2 * K_xy.mean()
    return F.relu(mmd_val)

def directionality_loss(pred, y, ctrl):
    """
    Calculates 1 - CosineSimilarity between (Pred - Ctrl) and (Y - Ctrl).
    Ensures the predicted expression shift is in the same 'direction' 
    (upregulation vs downregulation) as the ground truth.
    """
    # 1. Calculate the 'shift' vectors relative to control
    # ctrl shape is (Genes,), broadcasts to (Batch, Genes)
    delta_pred = pred - ctrl
    delta_y = y - ctrl

    # 2. Compute Cosine Similarity along the gene dimension (dim=1)
    # eps=1e-8 prevents division by zero if a vector has 0 magnitude
    cos_sim = F.cosine_similarity(delta_pred, delta_y, dim=1, eps=1e-8)

    # 3. Loss is 1 - similarity (we want sim to be 1.0)
    return 1.0 - cos_sim.mean()

def mmd(pred, y, perts, ctrl, dict_filter, direction_lambda=0.1):
    """
    Combined MMD + Directionality Loss module.
    
    Args:
        pred: (Batch, Genes)
        y: (Batch, Genes)
        perts: List/Array of perturbation labels
        ctrl: (Genes,) - The control/mean expression profile
        dict_filter: {pert_label: gene_indices}
        direction_lambda: Weight for the directionality loss
    """
    if pred.ndim == 1:
        pred = pred.unsqueeze(0)
        
    device = pred.device
    perts_arr = np.array(perts)
    unique_perts = np.unique(perts_arr)
    
    total_loss = torch.tensor(0.0, device=device)
    valid_groups = 0

    for p in unique_perts:
        # Boolean mask for current perturbation
        pert_mask = (perts_arr == p)
        
        if not np.any(pert_mask):
            continue

        # 1. Filtering Genes
        # Retrieve indices for this perturbation (or all genes if ctrl/missing)
        if p != "ctrl" and p in dict_filter:
            retain_idx = dict_filter[p]
        else:
            retain_idx = np.arange(pred.shape[1])

        # Slice data
        pred_p = pred[pert_mask][:, retain_idx]
        y_p = y[pert_mask][:, retain_idx]
        ctrl_p = ctrl[retain_idx]

        # 2. Compute MMD (Distribution Matching)
        loss_component = maximum_mean_discrepancy(pred_p, y_p)

        # 3. Compute Directionality (Vector Alignment)
        # Only apply if lambda > 0
        if direction_lambda > 0:
            # We want the shift from control to be consistent
            dir_loss = directionality_loss(pred_p, y_p, ctrl_p)
            loss_component = loss_component + (direction_lambda * dir_loss)

        total_loss = total_loss + loss_component
        valid_groups += 1

    if valid_groups > 0:
        return total_loss / valid_groups
    else:
        return total_loss

In [5]:

class PertData_(PertData):
    def prepare_split(self, split = 'simulation',
                      seed = 1,
                      train_gene_set_size = 0.75,
                      combo_seen2_train_frac = 0.75,
                      combo_single_split_test_set_fraction = 0.1,
                      test_perts = None,
                      only_test_set_perts = False,
                      test_pert_genes = None,
                      split_dict_path=None,
                      val_size=0.1):
        available_splits = ['simulation', 'simulation_single', 'combo_seen0',
                            'combo_seen1', 'combo_seen2', 'single', 'no_test',
                            'no_split', 'custom']
        if split not in available_splits:
            raise ValueError('currently, we only support ' + ','.join(available_splits))
        self.split = split
        self.seed = seed
        self.subgroup = None
        self.val_size = val_size

        if split == 'custom':
            try:
                with open(split_dict_path, 'rb') as f:
                    self.set2conditions = pickle.load(f)
            except:
                    raise ValueError('Please set split_dict_path for custom split')
            return

        self.train_gene_set_size = train_gene_set_size
        split_folder = os.path.join(self.dataset_path, 'splits')
        if not os.path.exists(split_folder):
            os.mkdir(split_folder)
        split_file = self.dataset_name + '_' + split + '_' + str(seed) + '_' \
                                       +  str(train_gene_set_size) + '.pkl'
        split_path = os.path.join(split_folder, split_file)

        if test_perts:
            split_path = split_path[:-4] + '_' + test_perts + '.pkl'

        if os.path.exists(split_path):
            print('here1')
            print_sys("Local copy of split is detected. Loading...")
            set2conditions = pickle.load(open(split_path, "rb"))
            if split == 'simulation':
                subgroup_path = split_path[:-4] + '_subgroup.pkl'
                subgroup = pickle.load(open(subgroup_path, "rb"))
                self.subgroup = subgroup
        else:
            print_sys("Creating new splits....")
            if test_perts:
                test_perts = test_perts.split('_')

            if split in ['simulation', 'simulation_single']:
                # simulation split
                DS = DataSplitter(self.adata, split_type=split)

                adata, subgroup = DS.split_data(train_gene_set_size = train_gene_set_size,
                                                combo_seen2_train_frac = combo_seen2_train_frac,
                                                seed=seed,
                                                test_perts = test_perts,
                                                only_test_set_perts = only_test_set_perts
                                               )
                subgroup_path = split_path[:-4] + '_subgroup.pkl'
                pickle.dump(subgroup, open(subgroup_path, "wb"))
                self.subgroup = subgroup

            elif split[:5] == 'combo':
                # combo perturbation
                split_type = 'combo'
                seen = int(split[-1])

                if test_pert_genes:
                    test_pert_genes = test_pert_genes.split('_')

                DS = DataSplitter(self.adata, split_type=split_type, seen=int(seen))
                adata = DS.split_data(test_size=combo_single_split_test_set_fraction,
                                      test_perts=test_perts,
                                      test_pert_genes=test_pert_genes,
                                      seed=seed)

            elif split == 'single':
                # single perturbation
                DS = DataSplitter(self.adata, split_type=split)
                adata = DS.split_data(test_size=combo_single_split_test_set_fraction,val_size=val_size,
                                      seed=seed)

            elif split == 'no_test':
                # no test set
                DS = DataSplitter(self.adata, split_type=split)
                adata = DS.split_data(seed=seed)

            elif split == 'no_split':
                # no split
                adata = self.adata
                adata.obs['split'] = 'test'

            set2conditions = dict(adata.obs.groupby('split').agg({'condition':
                                                        lambda x: x}).condition)
            set2conditions = {i: j.unique().tolist() for i,j in set2conditions.items()}
            pickle.dump(set2conditions, open(split_path, "wb"))
            print_sys("Saving new splits at " + split_path)

        self.set2conditions = set2conditions

        if split == 'simulation':
            print_sys('Simulation split test composition:')
            for i,j in subgroup['test_subgroup'].items():
                print_sys(i + ':' + str(len(j)))
        print_sys("Done!")


    def load(self, data_name=None, data_path=None):
        ## void return anyways
        print(data_name)
        super().load(data_name, data_path)
        ## finding out hvg index set
        # sc.pp.neighbors(self.adata, n_neighbors=15, use_rep='X')
        # sc.pp.highly_variable_genes(self.adata, n_top_genes=2000, subset=False, flavor='seurat_v3')
        # self.hvg_idx = self.adata.var['highly_variable'].to_numpy().nonzero()[0]
    def get_dataloader(self, batch_size, test_batch_size = None):
        """
        Get dataloaders for training and testing

        Parameters
        ----------
        batch_size: int
            Batch size for training
        test_batch_size: int
            Batch size for testing

        Returns
        -------
        dict
            Dictionary of dataloaders

        """
        if test_batch_size is None:
            test_batch_size = batch_size

        self.node_map = {x: it for it, x in enumerate(self.adata.var.gene_name)}
        self.gene_names = self.adata.var.gene_name

        # Create cell graphs
        cell_graphs = {}
        if self.split == 'no_split':
            i = 'test'
            cell_graphs[i] = []
            for p in self.set2conditions[i]:
                if p != 'ctrl':
                    cell_graphs[i].extend(self.dataset_processed[p])

            print_sys("Creating dataloaders....")
            # Set up dataloaders
            test_loader = DataLoader(cell_graphs['test'],
                                batch_size=batch_size, shuffle=False)

            print_sys("Dataloaders created...")
            return {'test_loader': test_loader}
        else:
            if self.split =='no_test':
                splits = ['train','val']
            else:
                splits = ['train','val','test']
            print(self.set2conditions)
            for i in splits:
                cell_graphs[i] = []
                if i in self.set2conditions:
                    for p in self.set2conditions[i]:
                        cell_graphs[i].extend(self.dataset_processed[p])
            # print(cell_graphs)
            print_sys("Creating dataloaders....")

            # Set up dataloaders
            if(len(cell_graphs["val"]) == 0):
                ## give a subset of entries to val from train.
                shuffled_list = cell_graphs["train"].copy()
                random.shuffle(shuffled_list)
                split_index = int(self.val_size*len(shuffled_list))
                cell_graphs["val"] = shuffled_list[:split_index]
                cell_graphs["train"] = shuffled_list[split_index:]



            train_loader = DataLoader(cell_graphs['train'],
                                batch_size=batch_size, shuffle=True, drop_last = True)
            if len(cell_graphs["val"])>0:
                val_loader = DataLoader(cell_graphs['val'],
                                    batch_size=batch_size, shuffle=True)
            else:
                val_loader = None
            if len(cell_graphs["test"])>0:
                test_loader = DataLoader(cell_graphs['val'],
                                    batch_size=batch_size, shuffle=True)
            else:
                test_loader = None

            if self.split !='no_test':
                test_loader = DataLoader(cell_graphs['test'],
                                batch_size=batch_size, shuffle=False)
                self.dataloader =  {'train_loader': train_loader,
                                    'val_loader': val_loader,
                                    'test_loader': test_loader}

            else:
                self.dataloader =  {'train_loader': train_loader,
                                    'val_loader': val_loader}
            print_sys("Done!")


In [6]:
from torch import nn
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import coalesce
class Gated_Addition(nn.Module):
    def __init__(self,vector_dim:int):
        super().__init__()
        self.gating = nn.Linear(2*vector_dim,vector_dim)
    def forward(self,x1,x2):
        g = self.gating(torch.concat((x1,x2),dim=-1))
        return g*x1 + (1-g)*x2

# lowrank_perturber_sage.py


class SAGEEncoder(nn.Module):
    """
    Small GraphSAGE node encoder.
    """
    def __init__(self, x_dim, hidden, out, layers=2, dropout=0.0, normalize=False):
        super().__init__()
        dims = [x_dim] + [hidden]*(layers-1) + [out]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i+1]) for i in range(len(dims)-1)])
        self.dropout = dropout
        self.normalize = normalize
        self.bns = nn.ModuleList([nn.BatchNorm1d(d) for d in dims[1:-1]]) if layers > 1 else None
        print(f"Expected Dimensions of activations through encoder: {dims}")
    def forward(self, x, edge_index):
        h = x
        L = len(self.convs)
        for i, conv in enumerate(self.convs):
            h = conv(h, edge_index)
            if i < L - 1:
                h = F.relu(h)
                if self.bns is not None:
                    h = self.bns[i](h)
                if self.dropout > 0:
                    h = F.dropout(h, p=self.dropout, training=self.training)
        if self.normalize:
            h = F.normalize(h, p=2, dim=-1,eps=1e-8)
        return h

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import coalesce
# Assuming SAGEEncoder is defined elsewhere
# from your_project.encoders import SAGEEncoder

class AttentionalPerturber(nn.Module):
    """
    Predicts edge deltas using a GAT-like additive attention mechanism.

      h = SAGE(x, edge_index)
      h_z = concat(h, z_broadcasted_to_nodes)
      
      score_u = AttnLinear_Left(h_z)
      score_v = AttnLinear_Right(h_z)
      
      Δ_{uv} = LeakyReLU(score_u[u] + score_v[v])
      w' = softplus( w + λ * Δ )
    """
    def __init__(self,
                 x_dim: int,
                 z_dim: int,
                 enc_hidden: int = 128,
                 enc_out: int = 128,
                 enc_layers: int = 2,
                 enc_dropout: float = 0.0,
                 enc_l2norm: bool = False,
                 nonneg: bool = True,
                 symmetric: bool = True,
                 lambda_init: float = 0.5):
        super().__init__()
        self.encoder = SAGEEncoder(x_dim, enc_hidden, enc_out,
                                   layers=enc_layers,
                                   dropout=enc_dropout,
                                   normalize=enc_l2norm)

        # Input to attention heads is node embedding + global z
        attn_in_dim = enc_out + z_dim
        
        # We use separate linear layers for source (left) and target (right)
        # This allows for asymmetric interactions (e.g., GATv2 style)
        self.attn_l = nn.Linear(attn_in_dim, 1, bias=False)
        self.attn_r = nn.Linear(attn_in_dim, 1, bias=False)
        self.leaky_relu = nn.LeakyReLU(0.2)

        self.nonneg = nonneg
        self.symmetric = symmetric
        self.lambda_scale = nn.Parameter(torch.tensor(lambda_init, dtype=torch.float32))
        
        # Note: We ignore the original weight `w` in this formulation,
        # but you could easily add it to `attn_in_dim` if needed
        # by first creating edge features. This is the simpler node-centric version.

    @staticmethod
    def _symmetrize(edge_index, w, reduce: str = "mean"):
        # (Copied from your original class)
        ei, w_sum = coalesce(edge_index, w, reduce="add")
        ones = torch.ones_like(w)
        _, count = coalesce(edge_index, ones, reduce="add")
        if reduce == "mean":
            w_sym = w_sum / count.clamp_min(1)
        elif reduce == "sum":
            w_sym = w_sum
        else:
            raise ValueError("reduce must be 'mean' or 'sum'")
        return ei, w_sym

    def forward(self,
                x: torch.Tensor,             # [N, x_dim]
                edge_index: torch.Tensor,     # [2, E]
                w: torch.Tensor,              # [E]
                z: torch.Tensor               # [z_dim]
                ):
        # 1) Node embeddings
        h = self.encoder(x, edge_index)       # [N, enc_out]
        N = h.size(0)

        # 2) Combine node features with global perturbation z
        z_b = z.expand(N, -1)                 # [N, z_dim]
        h_z = torch.cat([h, z_b], dim=-1)     # [N, enc_out + z_dim]

        # 3) Get attention scores for all nodes
        scores_l = self.attn_l(h_z)           # [N, 1]
        scores_r = self.attn_r(h_z)           # [N, 1]

        # 4) Compute edge deltas by summing endpoint scores
        u, v = edge_index[0], edge_index[1]   # [E]
        delta_scores = scores_l[u] + scores_r[v]  # [E, 1]
        delta = self.leaky_relu(delta_scores).squeeze(-1) # [E]

        # 5) Apply and post-process
        w_prime = w + self.lambda_scale * delta
        if self.nonneg:
            w_prime = F.softplus(w_prime)

        if self.symmetric:
            edge_index, w_prime = self._symmetrize(edge_index, w_prime, reduce="mean")
            _, delta = self._symmetrize(edge_index, delta, reduce="mean")
            return edge_index, w_prime, delta

        return edge_index, w_prime, delta
    

In [8]:
class GEARS_No_Coexpress(GEARS_Model):
    def __init__(self, args):
        super().__init__(args)
        self.layers_emb_pos = torch.nn.ModuleList() # Empty module list

In [9]:
class GEARS_No_Coexpress(GEARS_Model):
    def __init__(self, args):
        super().__init__(args)
        self.layers_emb_pos = torch.nn.ModuleList() # Empty module list

In [10]:
from torch_geometric.data import Data, Batch
class GEARS_MECH(GEARS_Model):
    def __init__(self, args):
        super().__init__(args)
        self.pretrain_phase = None
        hidden_size = args['hidden_size']
        self.return_graphs = False
        low_rank_percent = 0.2 ## make this something thats in the config.
        ## adding the gated expression embedding module
        self.expression_projection = nn.Linear(1,args["hidden_size"])
        self.gating = Gated_Addition(vector_dim=args["hidden_size"])
        ## adding self.graph_perturber
        r = int(low_rank_percent*self.G_coexpress.shape[1])
        print(f"Downprojected Rank {r}")
        ## it takes the gene-embeddings as an input
        ## i mean it uses the gener graph
        ## so x_dim is embedding_dim, ie: hidden_size in the args , same for the pert_dim ie: z_dim
        self.graph_perturber = AttentionalPerturber(
            x_dim=hidden_size, z_dim= hidden_size,
        )
        ## other args used in the above init:
        # enc_hidden: int = 128,
        # enc_out: int = 128,
        # enc_layers: int = 2,
        # enc_dropout: float = 0.0,
        # enc_l2norm: bool = False,
        # nonneg: bool = True,
        # symmetric: bool = True,
        # lambda_init: float = 0.5


        ## will have to be a module, that takes in an edge_weights, a learnable perturbation embedding, and outputs a changed/perturbed graph index of the same dimension.

    def forward(self, data):
        """
        Forward pass of the model
        """
        x, pert_idx = data.x, data.pert_idx
        num_graphs = len(data.batch.unique()) # each cell has its own graph
        ## get base gene embeddings, num_batch of the same graph.
        emb = self.gene_emb(torch.LongTensor(list(range(self.num_genes))).repeat(num_graphs, ).to(self.args['device']))
        emb = self.bn_emb(emb)
        base_emb = self.emb_trans(emb)

        ## EXPRESSION EMBEDDING MODULE, TO EMBED CELL EXPRESSION.
        x_cells = x.view(num_graphs, self.num_genes, 1)
        means   = x_cells.mean(dim=0, keepdim=True)  # (1, G, 1)
        centered= x_cells - means                    # (B, G, 1)
        # back to (B*G, 1) for linear
        centered = centered.view(-1, 1)
        expr_emb = self.expression_projection(centered)  # (B*G, H)
        base_emb = self.gating(base_emb,expr_emb)

        #
        if not self.pretrain_phase:
            ### START OF BLOCK FOR THIS
            pert_global_emb = self.pert_emb(torch.LongTensor(list(range(self.num_perts))).to(self.args['device']))
            ## augment global perturbation embedding with GNN
            ## pass the perturbation global embedding through gnn
            for idx, layer in enumerate(self.sim_layers):
                # GCN with Perturbation graph constructed via Gene-Ontology Network
                pert_global_emb = layer(pert_global_emb, self.G_sim, self.G_sim_weight)
                if idx < self.num_layers - 1:
                    pert_global_emb = pert_global_emb.relu()
            pert_index = []
            for idx, i in enumerate(pert_idx):
                for j in i:
                    if j != -1:
                        ## j = -1 is the control perturbation index, dunno why they arent appending it
                        ## idx indicates which cell, j corresponds to the perturbation number.
                        pert_index.append([idx, j])
            pert_index = torch.tensor(pert_index).T ## 2 x numgraohs with (cell_id, pert_id)
            base_emb = base_emb.reshape(num_graphs, self.num_genes, -1)

            if pert_index.shape[0] != 0:
                ### in case all samples in the batch are controls, then there is no indexing for pert_index.
                pert_track = {}
                ## i is index of that cell in the batch, j is the perturbation index/type ?
                ## pert_index[0] is the cell_id
                for i, j in enumerate(pert_index[0]):
                    if j.item() in pert_track:
                        ## to pert_track, add the perturbation embedding for that perturbation, which is in pert_index[1][i]
                        pert_track[j.item()] = pert_track[j.item()] + pert_global_emb[pert_index[1][i]]
                        ## pert_track[cell_id] = pert_track[cell_id] + pert_global[pert_id corresponding to cell_id ]
                    else:
                        pert_track[j.item()] = pert_global_emb[pert_index[1][i]]

                # print(f"This is pert_track{pert_track}")

                ## Edge index will remain the same, only the edge weights will change.

                all_edge_weights = {}

                if len(list(pert_track.values())) > 0:
                    if len(list(pert_track.values())) == 1:
                        # circumvent when batch size = 1 with single perturbation and cannot feed into MLP
                        emb_total = self.pert_fuse(torch.stack(list(pert_track.values()) * 2))
                    else:
                        ## pass this specific perturbation embedding set through an mlp for it to add to the base embedding
                        emb_total = self.pert_fuse(torch.stack(list(pert_track.values())))

                    for idx, j in enumerate(pert_track.keys()):
                        ## j is cell_id, ie: which graph we're adding this to, and emb_total[idx] has the perturbation embedding that we're adding to that corresponding cell
                        ##
                        node_embs = base_emb[j]
                        pert_emb = emb_total[idx]
                        edge_index, w_prime, delta = self.graph_perturber(node_embs, self.G_coexpress, self.G_coexpress_weight, pert_emb)
                        all_edge_weights[j] = w_prime

        pos_emb = self.emb_pos(torch.LongTensor(list(range(self.num_genes))).repeat(num_graphs, ).to(self.args['device']))
        data_list = []
        # print(f"This is all_edge_weights {all_edge_weights}")
        ## one hot vector to tell if its a control or not
        ctrl_no_ctrl = torch.zeros(num_graphs,1).to(self.args['device'])
        for i in range(num_graphs):
            node_features = pos_emb[i * self.num_genes : (i + 1) * self.num_genes]
            # print(f"Node feature dim: {node_features.shape}")
            if i in all_edge_weights:
                ## special graph corresponding to perturbed edges
                data_list.append(
                    Data(
                        x=node_features,  # Node features for graph i
                        edge_index=self.G_coexpress, # Same structure
                        edge_weight=all_edge_weights[i] # DIFFERENT weights
                    )
                )
                ctrl_no_ctrl[i] = 1.0
            else:
                ## control cells 
                data_list.append(
                    Data(
                        x=node_features,  # Node features for graph i
                        edge_index=self.G_coexpress, # Same structure
                        edge_weight= self.G_coexpress_weight
                    )
                )
        
        batch = Batch.from_data_list(data_list)
        batch = batch.to(self.args["device"])
        for idx, layer in enumerate(self.layers_emb_pos):
            batch.x = layer(batch.x, batch.edge_index, batch.edge_weight)
            if idx < len(self.layers_emb_pos) - 1:
                batch.x = batch.x.relu()
        final_pos_emb = batch.x.view(num_graphs, self.num_genes, -1)
        base_emb = base_emb + 0.7 * final_pos_emb
        base_emb = base_emb.reshape(num_graphs*self.num_genes,-1)
        base_emb = self.emb_trans_v2(base_emb)
        base_emb = base_emb.reshape(num_graphs * self.num_genes, -1)
        base_emb = self.bn_pert_base(base_emb)
        base_emb = self.transform(base_emb)
        out = self.recovery_w(base_emb)
        out = out.reshape(num_graphs, self.num_genes, -1)
        out = out.unsqueeze(-1) * self.indv_w1
        w = torch.sum(out, axis = 2)
        out = w + self.indv_b1
        cross_gene_embed = self.cross_gene_state(out.reshape(num_graphs, self.num_genes, -1).squeeze(2))
        cross_gene_embed = cross_gene_embed.repeat(1, self.num_genes)
        cross_gene_embed = cross_gene_embed.reshape([num_graphs,self.num_genes, -1])
        cross_gene_out = torch.cat([out, cross_gene_embed], 2)
        cross_gene_out = cross_gene_out * self.indv_w2
        cross_gene_out = torch.sum(cross_gene_out, axis=2)
        out = cross_gene_out + self.indv_b2
        out = out.reshape(num_graphs * self.num_genes, -1)  + x.reshape(-1,1) 
        out = torch.split(torch.flatten(out), self.num_genes)
        if self.return_graphs:
            graph_data =   {"graphs": data_list,
                            "graph_metadata": ctrl_no_ctrl
                            }
            return out, graph_data
            
        return torch.stack(out)
    def compute_graphs(self,data):
        self.return_graphs = True
        with torch.no_grad():
            out, graph_data = self.forward(data)
        self.return_graphs = False
        return out,graph_data


In [11]:
from gears import GEARS
import numpy as np
from gears.utils import loss_fct
from torch import nn
from gears.utils import  get_similarity_network,GeneSimNetwork
from gears.utils import loss_fct
from gears.inference import *
import numpy as np
from gears.inference import evaluate,compute_metrics
from copy import deepcopy
import pandas as pd
from gears.utils import np_pearson_cor
import networkx as nx
from functools import reduce
import operator
import torch
from functools import reduce
from tqdm import tqdm
import networkx as nx
import torch
import numpy as np
import pandas as pd
from functools import reduce
import operator
from tqdm.auto import tqdm # Import tqdm


class GeneSimNetworkKHops():
    """
    GeneSimNetwork class

    Args:
        edge_list (pd.DataFrame): edge list of the network
        gene_list (list): list of gene names
        node_map (dict): dictionary mapping gene names to node indices
    """
    def __init__(self, edge_list, gene_list, node_map):
        """
        Initialize GeneSimNetwork class
        """
        self.edge_list = edge_list
        self.gene_list = gene_list
        self.node_map = node_map
        self.G = nx.from_pandas_edgelist(self.edge_list, source='source',
                        target='target', edge_attr=['importance'],
                        create_using=nx.DiGraph())
        for n in self.gene_list:
            if n not in self.G.nodes():
                self.G.add_node(n)
        self._update_tensors()

    def _update_tensors(self):
        """Helper to regenerate tensors from the current nx.Graph state."""
        if not self.G.edges:
            self.edge_index = torch.empty((2, 0), dtype=torch.long)
            self.edge_weight = torch.empty((0,), dtype=torch.float)
            return

        # print(self.node_map)
        # print(self.G.edges)
        edge_index_ = [(self.node_map[e[0]], self.node_map[e[1]]) for e in self.G.edges]
        self.edge_index = torch.tensor(edge_index_, dtype=torch.long).T

        edge_attr = nx.get_edge_attributes(self.G, 'importance')
        importance = np.array([edge_attr[e] for e in self.G.edges])
        self.edge_weight = torch.Tensor(importance)

    # --- UPDATED METHOD ---

    def add_zero_weight_khop_edges(self, k, m):
        """
        For each node, finds all nodes reachable within k-hops. From this set,
        it calculates the max-multiplicative-strength path for each.
        It then adds 'm' zero-weight edges to the unconnected nodes with
        the highest strength.

        Args:
            k (int): The maximum hop distance to search.
            m (int): The number of new edges to add per node.
        """
        if k <= 0:
            print("k must be a positive integer.")
            return

        # Changed print statement
        print(f"Graph modification in progress. Original edge count: {len(self.edge_index[0])}")
        new_edges_to_add = []

        # Added tqdm wrapper to the main loop
        for start_node in tqdm(list(self.G.nodes()), desc="Processing nodes"):
            potential_edges = []

            # 1. OPTIMIZATION: Get the subgraph of all nodes reachable
            #    within k hops using nx.ego_graph.
            ego_graph = nx.ego_graph(self.G, n=start_node, radius=k)

            # 2. Iterate ONLY over this smaller set of reachable nodes
            for target_node in ego_graph.nodes():
                # Exclude self-loops and existing direct edges
                if start_node == target_node or self.G.has_edge(start_node, target_node):
                    continue

                # 3. Find all simple paths (this is the required slow part)
                #    We still search on the *original graph* (self.G)
                paths = nx.all_simple_paths(self.G,
                                            source=start_node,
                                            target=target_node,
                                            cutoff=k)

                max_strength = 0.0
                for path in paths:
                    if len(path) > 1:
                        # Calculate multiplicative strength
                        strength = reduce(operator.mul,
                                        (self.G[u][v]['importance'] for u, v in zip(path[:-1], path[1:])))
                        if strength > max_strength:
                            max_strength = strength

                # If a path was found, store this as a potential edge
                if max_strength > 0:
                    potential_edges.append((target_node, max_strength))

            # 4. Sort potential edges by their calculated strength
            potential_edges.sort(key=lambda x: x[1], reverse=True)

            # 5. Add the top 'm' new edges
            for target_node, _ in potential_edges[:m]:
                new_edges_to_add.append((start_node, target_node, {'importance': 0.0}))

        # 6. Add all new edges to the graph at once
        self.G.add_edges_from(new_edges_to_add)
        print(f"Added {len(new_edges_to_add)} new zero-weight edges.")

        # 7. Regenerate tensors to reflect the new graph structure
        self._update_tensors()
        print(f"Tensors updated. New edge count: {len(self.edge_index[0])}")

In [12]:
from gears.inference import deeper_analysis,non_dropout_analysis

class GEARS_PRETRAIN(GEARS):
    def __init__(self, pert_data, device='cuda', weight_bias_track=False, proj_name='GEARS', exp_name='GEARS'):

        self.weight_bias_track = weight_bias_track

        if self.weight_bias_track:
            import wandb
            wandb.init(project=proj_name, name=exp_name)
            self.wandb = wandb
        else:
            self.wandb = None

        self.device = device
        self.config = None

        self.dataloader = pert_data.dataloader ##
        self.adata = pert_data.adata
        self.node_map = pert_data.node_map
        self.node_map_pert = pert_data.node_map_pert
        self.data_path = pert_data.data_path
        self.dataset_name = pert_data.dataset_name
        self.split = pert_data.split
        self.seed = pert_data.seed
        self.train_gene_set_size = pert_data.train_gene_set_size
        self.set2conditions = pert_data.set2conditions
        self.subgroup = pert_data.subgroup
        self.gene_list = pert_data.gene_names.values.tolist()
        self.pert_list = pert_data.pert_names.tolist()
        self.num_genes = len(self.gene_list)
        self.num_perts = len(self.pert_list)
        self.default_pert_graph = pert_data.default_pert_graph
        self.saved_pred = {}
        self.saved_logvar_sum = {}

        self.ctrl_expression = torch.tensor(
            np.mean(self.adata.X[self.adata.obs.condition.values == 'ctrl'],
                    axis=0)).reshape(-1, ).to(self.device)
        pert_full_id2pert = dict(self.adata.obs[['condition_name', 'condition']].values)
        self.dict_filter = {pert_full_id2pert[i]: j for i, j in
                            self.adata.uns['non_zeros_gene_idx'].items() if
                            i in pert_full_id2pert}
        self.ctrl_adata = self.adata[self.adata.obs['condition'] == 'ctrl']

        gene_dict = {g:i for i,g in enumerate(self.gene_list)}
        self.pert2gene = {p: gene_dict[pert] for p, pert in
                            enumerate(self.pert_list) if pert in self.gene_list}
        self.hvg_idx = getattr(pert_data,"hvg_idx", None)

    def update(self,
                pert_data):
        ''' Function to update the pert_data the model is using, ie to switch from control pert_data to perturbed pert_data. '''
        self.dataloader = pert_data.dataloader ##
        self.adata = pert_data.adata
        self.node_map = pert_data.node_map
        self.node_map_pert = pert_data.node_map_pert
        self.data_path = pert_data.data_path
        self.dataset_name = pert_data.dataset_name
        self.split = pert_data.split
        self.seed = pert_data.seed
        self.train_gene_set_size = pert_data.train_gene_set_size
        self.set2conditions = pert_data.set2conditions
        self.subgroup = pert_data.subgroup
        self.gene_list = pert_data.gene_names.values.tolist()
        self.pert_list = pert_data.pert_names.tolist()
        self.num_genes = len(self.gene_list)
        self.num_perts = len(self.pert_list)
        self.default_pert_graph = pert_data.default_pert_graph
        self.saved_pred = {}
        self.saved_logvar_sum = {}
        self.ctrl_expression = torch.tensor(
            np.mean(self.adata.X[self.adata.obs.condition.values == 'ctrl'],
                    axis=0)).reshape(-1, ).to(self.device)
        pert_full_id2pert = dict(self.adata.obs[['condition_name', 'condition']].values)
        self.dict_filter = {pert_full_id2pert[i]: j for i, j in
                            self.adata.uns['non_zeros_gene_idx'].items() if
                            i in pert_full_id2pert}
        self.ctrl_adata = self.adata[self.adata.obs['condition'] == 'ctrl']

        gene_dict = {g:i for i,g in enumerate(self.gene_list)}
        self.pert2gene = {p: gene_dict[pert] for p, pert in
                            enumerate(self.pert_list) if pert in self.gene_list}
        self.hvg_idx = getattr(pert_data,"hvg_idx", None)


        ## calculating co expression similarity graph
        edge_list = {}
        edge_list["coexpress"] = get_similarity_network(network_type='co-express',
                                            adata=self.adata,
                                            threshold=self.graph_config["coexpress_threshold"],
                                            k=self.graph_config["num_similar_genes_co_express_graph"],
                                            data_path=self.data_path,
                                            data_name=self.dataset_name,
                                            split=self.split, seed=self.seed,
                                            train_gene_set_size=self.train_gene_set_size,
                                            set2conditions=self.set2conditions)
        ## checking if multi-hop
        if self.config["num_hops"] > 1:
            sim_network = GeneSimNetworkKHops(edge_list["coexpress"],self.pert_list,self.node_map)
            sim_network.add_zero_weight_khop_edges(
                k=self.config["num_hops"],
                m=self.config["num_add"],)
        else:
            sim_network = GeneSimNetwork(edge_list["coexpress"], self.pert_list, node_map = self.node_map_pert)
        self.config['G_coexpress'] = sim_network.edge_index
        self.config['G_coexpress_weight'] = sim_network.edge_weight
        edge_list["coexpress"] = get_similarity_network(network_type='go',
                                        adata=self.adata,
                                        threshold=self.graph_config["coexpress_threshold"],
                                        k=self.graph_config["num_similar_genes_go_graph"],
                                        pert_list=self.pert_list,
                                        data_path=self.data_path,
                                        data_name=self.dataset_name,
                                        split=self.split, seed=self.seed,
                                        train_gene_set_size=self.train_gene_set_size,
                                        set2conditions=self.set2conditions,
                                        default_pert_graph=self.default_pert_graph)

        self.config['G_go'] = sim_network.edge_index
        self.config['G_go_weight'] = sim_network.edge_weight




    def model_initialize(self, hidden_size = 64,
                            num_go_gnn_layers = 1,
                            num_gene_gnn_layers = 1,
                            decoder_hidden_size = 16,
                            num_similar_genes_go_graph = 20,
                            num_similar_genes_co_express_graph = 20,
                            coexpress_threshold = 0.4,
                            uncertainty = False,
                            uncertainty_reg = 1,
                            direction_lambda = 1e-1,
                            G_go = None,
                            G_go_weight = None,
                            G_coexpress = None,
                            G_coexpress_weight = None,
                            no_perturb = False,
                            gears_model=0,
                            num_hops=1,
                            num_add=1,
                            num_heads=4,
                            gated=False,
                            **kwargs
                        ):

        """
        Initialize the model

        Parameters
        ----------
        hidden_size: int
            hidden dimension, default 64
        num_go_gnn_layers: int
            number of GNN layers for GO graph, default 1
        num_gene_gnn_layers: int
            number of GNN layers for co-expression gene graph, default 1
        decoder_hidden_size: int
            hidden dimension for gene-specific decoder, default 16
        num_similar_genes_go_graph: int
            number of maximum similar K genes in the GO graph, default 20
        num_similar_genes_co_express_graph: int
            number of maximum similar K genes in the co expression graph, default 20
        coexpress_threshold: float
            pearson correlation threshold when constructing coexpression graph, default 0.4
        uncertainty: bool
            whether or not to turn on uncertainty mode, default False
        uncertainty_reg: float
            regularization term to balance uncertainty loss and prediction loss, default 1
        direction_lambda: float
            regularization term to balance direction loss and prediction loss, default 1
        G_go: scipy.sparse.csr_matrix
            GO graph, default None
        G_go_weight: scipy.sparse.csr_matrix
            GO graph edge weights, default None
        G_coexpress: scipy.sparse.csr_matrix
            co-expression graph, default None
        G_coexpress_weight: scipy.sparse.csr_matrix
            co-expression graph edge weights, default None
        no_perturb: bool
            predict no perturbation condition, default False
        gears_model: int
            0- original model, 1- expression embedding, 2 - GAT, 3 - TransformerConv, 4- No Coexpression 5- No perturbation.
        num_hops:int
            1- No additional zero weight edges will be added, else num_hops represents the max number of hops a target node is away from the source node, for us to consider adding a zero-weight edge between source-target
        num_add:int,
            1- Max number of edges of the above type we add for each node
        gated: bool
            Boolean expression that represents as to whether we're gating the expression embedding being added to the base embedding.

        Returns
        -------
        None
        """

        self.config = {
            'hidden_size': hidden_size,
            'num_go_gnn_layers' : num_go_gnn_layers,
            'num_gene_gnn_layers' : num_gene_gnn_layers,
            'decoder_hidden_size' : decoder_hidden_size,
            'num_similar_genes_go_graph' : num_similar_genes_go_graph,
            'num_similar_genes_co_express_graph' : num_similar_genes_co_express_graph,
            'coexpress_threshold': coexpress_threshold,
            'uncertainty' : uncertainty,
            'uncertainty_reg' : uncertainty_reg,
            'direction_lambda' : direction_lambda,
            'G_go': G_go,
            'G_go_weight': G_go_weight,
            'G_coexpress': G_coexpress,
            'G_coexpress_weight': G_coexpress_weight,
            'device': self.device,
            'num_genes': self.num_genes,
            'num_perts': self.num_perts,
            'no_perturb': no_perturb,
            'gears_model': gears_model,
            "num_hops":num_hops,
            "num_add":num_add,
            'num_heads': num_heads,
            'gated':gated
        }

        self.graph_config = {
            "threshold":coexpress_threshold,
            "k_coexpress":num_similar_genes_co_express_graph,
            "k_go":num_similar_genes_go_graph
        }
        if self.wandb:
            self.wandb.config.update(self.config)

        if self.config['G_coexpress'] is None:
            ## calculating co expression similarity graph
            edge_list = get_similarity_network(network_type='co-express',
                                                adata=self.adata,
                                                threshold=coexpress_threshold,
                                                k=num_similar_genes_co_express_graph,
                                                data_path=self.data_path,
                                                data_name=self.dataset_name,
                                                split=self.split, seed=self.seed,
                                                train_gene_set_size=self.train_gene_set_size,
                                                set2conditions=self.set2conditions)
            ## checking if multi-hop
            if self.config["num_hops"] > 1:
                sim_network = GeneSimNetworkKHops(edge_list,self.pert_list,self.node_map)
                sim_network.add_zero_weight_khop_edges(
                    k=self.config["num_hops"],
                    m=self.config["num_add"],)
            else:
                sim_network = GeneSimNetwork(edge_list, self.pert_list, node_map = self.node_map)

            # sim_network = GeneSimNetwork(edge_list, self.gene_list, node_map = self.node_map)
            self.config['G_coexpress'] = sim_network.edge_index
            self.config['G_coexpress_weight'] = sim_network.edge_weight

        if self.config['G_go'] is None:
            ## calculating gene ontology similarity graph
            edge_list = get_similarity_network(network_type='go',
                                                adata=self.adata,
                                                threshold=coexpress_threshold,
                                                k=num_similar_genes_go_graph,
                                                pert_list=self.pert_list,
                                                data_path=self.data_path,
                                                data_name=self.dataset_name,
                                                split=self.split, seed=self.seed,
                                                train_gene_set_size=self.train_gene_set_size,
                                                set2conditions=self.set2conditions,
                                                default_pert_graph=self.default_pert_graph)

            self.config['G_go'] = sim_network.edge_index
            self.config['G_go_weight'] = sim_network.edge_weight

        if self.config["gears_model"] == 0 :
            self.model = GEARS_Model(self.config).to(self.device)
        elif self.config["gears_model"] == 1:
            self.model = GEARS_MECH(self.config).to(self.device)
        #     self.model = GEARS_EMBED(self.config).to(self.device)
        # elif self.config["gears_model"] == 2:
        #     self.model = GEARS_GAT(self.config).to(self.device)
        # elif self.config["gears_model"] == 3:
        #     self.model = GEARS_Transformer(self.config).to(self.device)
        elif self.config["gears_model"] == 4:
            self.model = GEARS_No_Coexpress(self.config).to(self.device)
        # elif self.config["gears_model"] == 5:
        #     self.model = GEARS_No_Perturb(self.config).to(self.device)
        # elif self.config["gears_model"] == 6:
        #     self.model = GEARS_SelfAttn(self.config).to(self.device)
        # elif self.config["gears_model"] == 7:
        #     self.model = GEARS_CELL(self.config).to(self.device)
        # elif self.config["gears_model"] == 8:
        #     self.model = VariantTiny(self.config).to(self.device)
        # elif self.config["gears_model"] == 9:
        #     self.model = Variant(self.config).to(self.device)

        self.best_model = deepcopy(self.model)

    def pretrain_ctrl(self, epochs = 20,
                lr = 1e-3,
                weight_decay = 5e-4
                ):
        train_loader = self.dataloader['train_loader']
        val_loader = self.dataloader['val_loader']

        print(f"Length of train_loader: {len(train_loader)}")
        print(f"Length of val_loader {len(val_loader)}")

        ## This will change the model's forward, to reconstruction only

        self.model.pretrain_phase = True

        self.model = self.model.to(self.device)
        best_model = deepcopy(self.model)
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay = weight_decay)
        scheduler = StepLR(optimizer, step_size=1, gamma=0.5)
        min_val = np.inf
        print_sys('Start Training...')
        hvg_idx = self.hvg_idx
        for epoch in range(epochs):
            self.model.train()
            for step, batch in enumerate(train_loader):
                batch.to(self.device)
                optimizer.zero_grad()
                y = batch.y
                x = batch.x
                pred = self.model(batch)
                ## autoencoder style loss with directionality enforced.
                ## Consider switching to MSE instead ?
                loss = loss_fct(pred, x, batch.pert,
                                ctrl = self.ctrl_expression,
                                dict_filter = self.dict_filter,
                                direction_lambda = self.config['direction_lambda'])
                loss.backward()
                nn.utils.clip_grad_value_(self.model.parameters(), clip_value=1.0)
                optimizer.step()
                if self.wandb:
                    self.wandb.log({'training_loss': loss.item()})
                if step % 50 == 0:
                    log = "Epoch {} Step {} Train Loss: {:.4f}"
                    print_sys(log.format(epoch + 1, step + 1, loss.item()))
            scheduler.step()
            train_run = evaluate(train_loader,self.model,False,self.device)
            val_run = evaluate(val_loader,self.model,False,self.device)
            ## computing metrics :
            try:
                train_metrics, _ = compute_metrics(train_run)
                val_metrics, _ = compute_metrics(val_run)
                ## print epoch performance
                log = "Epoch {}: Train Overall MSE: {:.4f} " \
                        "Validation Overall MSE: {:.4f}. "
                print_sys(log.format(epoch + 1, train_metrics['mse'],
                                    val_metrics['mse']))
                if(min_val > val_metrics["mse"]):
                    print(f"New Best model has val_mse: {val_metrics['mse']}")
                    self.best_model = deepcopy(self.model)
            except Exception as e:
                print(f"Error while computing metrics: {e}")
            ## computing loss on the highly variable gene set.
            ## computing validation loss of the validation dataloader ?
            ## consider adding evaluation for reconstruction via the autoencoder over here ?
        print(f"Best Val Loss: {min_val}")
        print("Done Training ....")

    def train(
        self, epochs = 20,
        lr = 1e-3,
        weight_decay = 5e-4,
        use_mmd=False,):
        print("Training Phase with perturbing graph .....")

        if use_mmd:
            loss_used = mmd
        else:
            loss_used = loss_fct
        self.model.pretrain_phase = False

        train_loader = self.dataloader['train_loader']
        val_loader = self.dataloader['val_loader']
        print(f"Length of train_loader: {len(train_loader)}")
        print(f"Length of val_loader {len(val_loader)}")
        self.model = self.model.to(self.device)
        best_model = deepcopy(self.model)
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay = weight_decay)
        scheduler = StepLR(optimizer, step_size=1, gamma=0.5)
        min_val = np.inf
        print_sys('Start Training...')
        hvg_idx = self.hvg_idx
        for epoch in range(epochs):
            self.model.train()
            for step, batch in enumerate(train_loader):
                batch.to(self.device)
                optimizer.zero_grad()
                y = batch.y
                x = batch.x
                pred = self.model(batch)
                ## autoencoder style loss with directionality enforced.
                ## Consider switching to MSE instead ?
                loss = loss_used(pred, y, batch.pert,
                                ctrl = self.ctrl_expression,
                                dict_filter = self.dict_filter,
                                direction_lambda = self.config['direction_lambda'])
                loss.backward()
                nn.utils.clip_grad_value_(self.model.parameters(), clip_value=1.0)
                optimizer.step()
                if self.wandb:
                    self.wandb.log({'training_loss': loss.item()})
                if step % 50 == 0:
                    log = "Epoch {} Step {} Train Loss: {:.4f}"
                    print_sys(log.format(epoch + 1, step + 1, loss.item()))
            scheduler.step()
            train_run = evaluate(train_loader,self.model,False,self.device)
            val_run = evaluate(val_loader,self.model,False,self.device)
            ## computing metrics :
            try:
                train_metrics, _ = compute_metrics(train_run)
                val_metrics, _ = compute_metrics(val_run)
                log = "Epoch {}: Train Overall MSE: {:.4f} " \
                        "Validation Overall MSE: {:.4f}. "
                print_sys(log.format(epoch + 1, train_metrics['mse'],
                                    val_metrics['mse']))
                if(min_val > val_metrics["mse"]):
                    print(f"New Best model has val_mse: {val_metrics['mse']}")
                    min_val = val_metrics["mse"]
                    self.best_model = deepcopy(self.model)
            except Exception as e:
                print(f"Error while computing metrics: {e}")
            ## computing loss on the highly variable gene set.
            ## computing validation loss of the validation dataloader ?
            ## consider adding evaluation for reconstruction via the autoencoder over here

        ### TODO: Add test metrics.
        test_loader = self.dataloader['test_loader']
        print_sys("Start Testing...")
        test_res = evaluate(test_loader, self.best_model,
                            self.config['uncertainty'], self.device)
        test_metrics, test_pert_res = compute_metrics(test_res)    
        log = "Best performing model: Test Top 20 DE MSE: {:.4f}"
        print_sys(log.format(test_metrics['mse_de']))
        out = deeper_analysis(self.adata, test_res)
        out_non_dropout = non_dropout_analysis(self.adata, test_res)
        metrics = ['pearson_delta']
        # accumulate sums and counts per metric
        metric_sums = {}
        metric_counts = {}
        
        for pert, metrics in out.items():
            for metric, value in metrics.items():
                metric_sums[metric] = metric_sums.get(metric, 0) + value
                metric_counts[metric] = metric_counts.get(metric, 0) + 1

        # compute averages
        metric_avgs = {m: metric_sums[m] / metric_counts[m] for m in metric_sums}

        print(metric_avgs)



        print(f"Best Val Loss: {min_val}")
        print("Done Training ....")
        return out,out_non_dropout
    def _load_ground_truth_graphs(self, 
                                  base_path="/kaggle/input/coexpressiongraphs/Downloads/coexpression_graphs", 
                                  filename="edges.csv"):
        """
        Loads ground truth co-expression graphs from CSV files.

        This method builds a dictionary: {pert_name: 1D_weight_tensor}
        
        CRITICAL ASSUMPTIONS:
        - `self.node_map_gene`: A dict {'gene_name': gene_idx} must exist.
        - `self.model.G_coexpress`: The edge_index tensor [2, num_edges] must exist.
        - `self.node_map_pert`: A dict {'pert_name': pert_idx} must exist.
        - Files are located at: {base_path}/pert_ctrl+{pert_name}/{filename}
        """
        print("Attempting to load ground truth co-expression graphs...")
        
        # --- 1. Check for required attributes ---
        if not hasattr(self, 'node_map'):
            print("  Error: `self.node_map_gene` not found. Cannot load ground truth graphs.")
            return {}
        if not hasattr(self, 'model') or not hasattr(self.model, 'G_coexpress'):
            print("  Error: `self.model.G_coexpress` (edge_index) not found. Cannot load ground truth graphs.")
            return {}
        if not hasattr(self, 'node_map_pert'):
            print("  Error: `self.node_map_pert` not found. Cannot load ground truth graphs.")
            return {}

        # --- 2. Build Edge Index Map for fast lookup ---
        # This maps (u_idx, v_idx) -> position in the edge_weight tensor
        try:
            edge_index = self.model.G_coexpress.cpu()
            num_edges = edge_index.shape[1]
            edge_index_map = {}
            for i in range(num_edges):
                u_idx = edge_index[0, i].item()
                v_idx = edge_index[1, i].item()
                edge_index_map[(u_idx, v_idx)] = i
            
            gene_name_to_idx = self.node_map_gene
            print(f"  Built edge index map for {num_edges} edges.")
        except Exception as e:
            print(f"  Error building edge_index_map: {e}. Aborting GT load.")
            return {}

        # --- 3. Get all perturbation names, including 'ctrl' ---
        # (Based on the logic in your original function)
        pert_names = list(self.node_map_pert.keys())
        if 'ctrl' not in pert_names:
            pert_names.append('ctrl')

        gt_graph_store = {}
        
        # --- 4. Iterate and load each GT graph file ---
        for pert_name in pert_names:
            # Format: coexpression_graphs/pert_ctrl+GATA1/edges.csv
            file_path = os.path.join(base_path, f"pert_{pert_name}", filename)
            
            if not os.path.exists(file_path):
                print(f"  Info: No GT graph file found for '{pert_name}' at {file_path}. Skipping.")
                continue

            try:
                # Initialize a zero-vector for this GT graph's weights
                gt_weights_tensor = torch.zeros(num_edges, device=self.device)
                
                with open(file_path, 'r', encoding='utf-8') as f:
                    reader = csv.reader(f)
                    header = next(reader) # Skip header
                    
                    edges_found = 0
                    edges_not_in_index = 0
                    
                    for row in reader:
                        if len(row) < 3: continue # Skip malformed rows
                        source_name, target_name, weight_str = row[0], row[1], row[2]
                        
                        u_idx = gene_name_to_idx.get(source_name)
                        v_idx = gene_name_to_idx.get(target_name)
                        
                        if u_idx is None or v_idx is None:
                            continue # Gene name not in our map
                        
                        try:
                            weight = float(weight_str)
                        except ValueError:
                            continue # Invalid weight
                        
                        # Check for the edge in our model's edge_index
                        edge_pos = edge_index_map.get((u_idx, v_idx))
                        
                        if edge_pos is not None:
                            gt_weights_tensor[edge_pos] = weight
                            edges_found += 1
                        else:
                            # Optional: Check for reverse edge if graph is undirected
                            edge_pos_rev = edge_index_map.get((v_idx, u_idx))
                            if edge_pos_rev is not None:
                                gt_weights_tensor[edge_pos_rev] = weight
                                edges_found += 1
                            else:
                                edges_not_in_index += 1
                
                gt_graph_store[pert_name] = gt_weights_tensor.detach()
                print(f"  Loaded GT graph for '{pert_name}'. "
                      f"Mapped {edges_found} edges. ({edges_not_in_index} edges from file were not in model's G_coexpress).")

            except Exception as e:
                print(f"  Error loading GT graph for '{pert_name}' from {file_path}: {e}")

        print(f"Successfully loaded {len(gt_graph_store)} ground truth graphs.")
        return gt_graph_store

    # --- NEW HELPER METHOD 2: Pearson Correlation ---
    
    def calculate_pearson(self, x, y):
        """Calculates the Pearson correlation coefficient between two 1D tensors."""
        # Center the vectors
        x_centered = x - torch.mean(x)
        y_centered = y - torch.mean(y)
        
        # Use cosine similarity on centered vectors
        # Add epsilon for numerical stability in case of zero variance
        epsilon = 1e-8
        cos_sim = F.cosine_similarity(x_centered.unsqueeze(0), 
                                      y_centered.unsqueeze(0), 
                                      eps=epsilon)
        return cos_sim.item()

    # --- UPDATED Main Function ---
    
    def look_cell_graphs(self, dataloader):
        """
        Analyzes and compares predicted graph edge weights for different
        perturbations against:
        1. The baseline control graph.
        2. The corresponding ground truth co-expression graph.
        
        Assumes:
        - self.model.G_coexpress_weight: The baseline control edge weights (1D Tensor).
        - self.model.compute_graphs() returns: (pred, {"graphs": [Data, Data, ...]}).
        - The input `batch.pert` corresponds 1-to-1 with the list of output graphs.
        - `self.node_map_gene` exists for loading GT graphs.
        """
        
        print("Starting graph comparison analysis...")
        
        # --- 1. Get Baseline Control Graph ---
        try:
            control_weights = self.model.G_coexpress_weight.to(self.device).detach()
            num_edges = control_weights.shape[0]
            
            if self.model.G_coexpress_weight.shape[0] != self.model.G_coexpress.shape[1]:
                 print(f"Warning: Control weights shape ({control_weights.shape[0]}) "
                       f"does not match edge_index shape ({self.model.G_coexpress.shape[1]}).")

            print(f"Loaded baseline control graph with {num_edges} edges.")
            
        except AttributeError:
            print("Error: `self.model.G_coexpress_weight` or `self.model.G_coexpress` not found. Aborting.")
            return
        
        # --- 1.5. [NEW] Load Ground Truth Graphs ---
        # This calls the new helper method and stores GT graphs in a dictionary
        # self.gt_graph_store = { 'pert_name_1': tensor, 'pert_name_2': tensor, ... }
        try:
            # This assumes _load_ground_truth_graphs is a method of the same class
            self.gt_graph_store = self._load_ground_truth_graphs() 
        except Exception as e:
            print(f"Error during Ground Truth graph loading: {e}. Proceeding without GT comparison.")
            self.gt_graph_store = {} # Ensure it's a dict
        
        # --- 2. Create Perturbation Name Map ---
        try:
            idx_to_pert_name = {v: k for k, v in self.node_map_pert.items()}
            idx_to_pert_name[-1] = 'ctrl'
        except AttributeError:
            print("Error: `self.node_map_pert` not found. Aborting.")
            return

        # Dictionary to store aggregated results
        results_store = {}

        # --- 3. Iterate Through Dataloader ---
        for step, batch in enumerate(dataloader):
            with torch.no_grad():
                batch.to(self.device)
                
                # Run the model
                pred, graph_data = self.model.compute_graphs(batch)

                # --- 4. Get Predicted Graphs and Labels ---
                predicted_graphs_list = graph_data.get("graphs")
                if predicted_graphs_list is None or not isinstance(predicted_graphs_list, list):
                    print(f"Error: 'graphs' key not in model output or is not a list. Aborting batch {step}.")
                    continue
                
                num_graphs_in_batch = len(predicted_graphs_list)
                pert_indices = np.array(batch.pert) 

                if num_graphs_in_batch != pert_indices.shape[0]:
                    print(f"Warning: Mismatch in batch {step}. "
                          f"Model output {num_graphs_in_batch} graphs, but input batch had {pert_indices.shape[0]} labels. Skipping.")
                    continue

                # --- 5. Compare Each Graph in Batch ---
                for i in range(num_graphs_in_batch):
                    
                    pert_idx = pert_indices[i].item()
                    pert_name = idx_to_pert_name.get(pert_idx, f"Unknown_idx_{pert_idx}")
                    
                    data_graph_i = predicted_graphs_list[i]
                    predicted_weights_graph_i = data_graph_i.edge_weight.detach()
                    
                    if predicted_weights_graph_i.shape[0] != num_edges:
                        print(f"Warning: Shape mismatch in batch {step}, graph {i} (pert: {pert_name}). "
                              f"Expected {num_edges} edges, but predicted graph has {predicted_weights_graph_i.shape[0]}. Skipping graph.")
                        continue
                    
                    # --- 5a. Comparison vs. Control (Original Logic) ---
                    
                    edge_diff = predicted_weights_graph_i - control_weights
                    mean_abs_diff = torch.mean(torch.abs(edge_diff)).item()
                    
                    epsilon = 1e-6
                    control_is_zero = torch.abs(control_weights) < epsilon
                    pred_is_zero = torch.abs(predicted_weights_graph_i) < epsilon
                    
                    new_edges_mask = control_is_zero & (~pred_is_zero)
                    num_new_edges = torch.sum(new_edges_mask).item()
                    
                    avg_new_edge_strength = 0.0
                    if num_new_edges > 0:
                        avg_new_edge_strength = torch.mean(torch.abs(predicted_weights_graph_i[new_edges_mask])).item()

                    lost_edges_mask = (~control_is_zero) & pred_is_zero
                    num_lost_edges = torch.sum(lost_edges_mask).item()

                    # --- 5b. [NEW] Comparison vs. Ground Truth ---
                    
                    # Get the pre-loaded GT graph tensor for this perturbation
                    gt_weights = self.gt_graph_store.get(pert_name)
                    
                    gt_pearson_corr = np.nan
                    gt_cos_sim = np.nan
                    
                    if gt_weights is not None:
                        # Check for shape mismatch (should not happen if loading was correct)
                        if gt_weights.shape[0] == num_edges:
                            # Use new helper function for Pearson
                            gt_pearson_corr = self.calculate_pearson(predicted_weights_graph_i, gt_weights)
                            
                            # Calculate Cosine Similarity
                            gt_cos_sim = F.cosine_similarity(
                                predicted_weights_graph_i.unsqueeze(0), 
                                gt_weights.unsqueeze(0)
                            ).item()
                        else:
                            print(f"Warning: GT graph for {pert_name} has shape {gt_weights.shape[0]} but expected {num_edges}. Skipping GT comparison.")
                    # else: No GT graph was loaded for this pert_name, metrics will remain np.nan

                    # --- 6. Store Results (Updated) ---
                    if pert_name not in results_store:
                        results_store[pert_name] = {
                            'count': 0,
                            'mean_abs_diff': [],
                            'num_new_edges': [],
                            'avg_new_edge_strength': [],
                            'num_lost_edges': [],
                            'gt_pearson_corr': [],  # New
                            'gt_cos_sim': []        # New
                        }
                    
                    results_store[pert_name]['count'] += 1
                    results_store[pert_name]['mean_abs_diff'].append(mean_abs_diff)
                    results_store[pert_name]['num_new_edges'].append(num_new_edges)
                    if num_new_edges > 0:
                        results_store[pert_name]['avg_new_edge_strength'].append(avg_new_edge_strength)
                    results_store[pert_name]['num_lost_edges'].append(num_lost_edges)
                    
                    # Append GT metrics (will append np.nan if not found)
                    results_store[pert_name]['gt_pearson_corr'].append(gt_pearson_corr)
                    results_store[pert_name]['gt_cos_sim'].append(gt_cos_sim)

        # --- 7. Print Final Aggregated Report (Updated) ---
        print("\n--- 📊 Graph Comparison Report ---")
        print(f"Analyzed {sum(v['count'] for v in results_store.values())} total graphs across {len(results_store)} perturbations.")
        
        for pert_name in sorted(results_store.keys(), key=lambda x: (x == 'ctrl', x)): # Sort, put 'ctrl' first
            stats = results_store[pert_name]
            count = stats['count']
            
            # --- Original Averages ---
            avg_mean_diff = np.mean(stats['mean_abs_diff'])
            avg_new_edges = np.mean(stats['num_new_edges'])
            avg_lost_edges = np.mean(stats['num_lost_edges'])
            
            if stats['avg_new_edge_strength']:
                avg_new_edge_strength = np.mean(stats['avg_new_edge_strength'])
            else:
                avg_new_edge_strength = 0.0

            # --- [NEW] GT Averages ---
            # Use np.nanmean to safely ignore 'nan' values if a GT graph was missing
            avg_gt_pearson = np.nanmean(stats['gt_pearson_corr'])
            avg_gt_cos_sim = np.nanmean(stats['gt_cos_sim'])

            print(f"\n**Perturbation: {pert_name} (n={count})**")
            print(f"  --- vs. Control Graph ---")
            print(f"    Avg. Mean Absolute Edge Diff: {avg_mean_diff:.5f}")
            print(f"    Avg. New Edges Created (0 -> non-0): {avg_new_edges:.2f}")
            print(f"    Avg. Strength of New Edges: {avg_new_edge_strength:.5f}")
            print(f"    Avg. Edges Lost (non-0 -> 0): {avg_lost_edges:.2f}")
            
            print(f"  --- vs. Ground Truth Graph ---")
            # Only print GT stats if they are not all 'nan'
            if not np.isnan(avg_gt_pearson):
                print(f"    Avg. Pearson Correlation: {avg_gt_pearson:.4f}")
                print(f"    Avg. Cosine Similarity: {avg_gt_cos_sim:.4f}")
            else:
                print(f"    (No Ground Truth graph loaded for this perturbation)")

        print("-----------------------------------")
        return results_store

  

In [13]:
!mkdir -p data
!mkdir model_output

In [14]:
import scanpy as sc
from gears.utils import zip_data_download_wrapper
def data_download(data_name,data_dir,extract_dir):
    if data_name == 'norman':
        url = 'https://dataverse.harvard.edu/api/access/datafile/6154020'
    elif data_name == 'adamson':
        url = 'https://dataverse.harvard.edu/api/access/datafile/6154417'
    elif data_name == 'dixit':
        url = 'https://dataverse.harvard.edu/api/access/datafile/6154416'
    elif data_name == 'replogle_k562_essential':
        ## Note: This is not the complete dataset and has been filtered
        url = 'https://dataverse.harvard.edu/api/access/datafile/7458695'
    elif data_name == 'replogle_rpe1_essential':
        ## Note: This is not the complete dataset and has been filtered
        url = 'https://dataverse.harvard.edu/api/access/datafile/7458694'
    else:
        print("None of these datasets exist")
        return
    zip_data_download_wrapper(url, data_dir, extract_dir)


DATA_ROOT="data"
DATASET_NAME="norman"
FNAME=os.path.join(DATA_ROOT,DATASET_NAME,"perturb_processed.h5ad")
# CTRL_NAME="ctrl"

In [15]:
# data_download(data_name= DATASET_NAME, data_dir= os.path.join(DATA_ROOT,DATASET_NAME),extract_dir = DATA_ROOT)
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ctrl_dir = os.path.join(DATA_ROOT,"ctrl" + DATASET_NAME)
# adata = sc.read_h5ad(FNAME)
# ctrl_adata = adata[adata.obs['condition']==CTRL_NAME]
# os.makedirs(ctrl_dir ,exist_ok=True)
# ctrl_adata.write_h5ad(filename=os.path.join(ctrl_dir, "perturb_processed.h5ad"))

# ## creating symlinks to all other files in DATA_ROOT/DATASET_NAME/ to ctrl_dir, with excpetion of perturb_processed.h5ad
# for f in os.listdir(os.path.join(DATA_ROOT,DATASET_NAME)):
#     if f!="perturb_processed.h5ad":
#         if not os.path.exists(os.path.join(ctrl_dir,f)):
#             os.symlink(os.path.abspath(os.path.join(DATA_ROOT,DATASET_NAME,f)), os.path.join(ctrl_dir,f))

In [16]:
pert_data = PertData_(data_path=DATA_ROOT)
pert_data.load(data_name = DATASET_NAME,data_path = os.path.join(DATA_ROOT,DATASET_NAME))
pert_data.prepare_split(split = 'single', seed = 1,combo_single_split_test_set_fraction=0.3)
pert_data.get_dataloader(batch_size=32,test_batch_size=128)

Downloading...
100%|██████████| 9.46M/9.46M [00:00<00:00, 16.9MiB/s]
Downloading...


norman


100%|██████████| 169M/169M [00:06<00:00, 27.2MiB/s]
Extracting zip file...
Done!
Downloading...
100%|██████████| 559k/559k [00:00<00:00, 2.24MiB/s]
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Creating pyg object for each cell in the data...
Creating dataset file...
100%|██████████| 277/277 [04:54<00:00,  1.06s/it]
Done!
Saving new dataset pyg object at data/norman/data_pyg/cell_graphs.pkl
Done!
Creating new splits....
Saving new splits at data/norman/splits/norman_single_1_0.75.pkl
Done!
Creating dataloaders....
Done!


{'test': ['TSC22D1+ctrl', 'MAML2+ctrl', 'DUSP9+ctrl', 'BCORL1+ctrl', 'MEIS1+ctrl', 'CBL+ctrl', 'ctrl+SET', 'SLC4A1+ctrl', 'ZNF318+ctrl', 'COL2A1+ctrl', 'MAP4K5+ctrl', 'UBASH3B+ctrl', 'AHR+ctrl', 'S1PR2+ctrl', 'ctrl+CNN1', 'CELF2+ctrl', 'MAP4K3+ctrl', 'CDKN1A+ctrl', 'ctrl+MEIS1', 'MAPK1+ctrl', 'C19orf26+ctrl', 'ctrl+UBASH3B', 'CKS1B+ctrl', 'PRTG+ctrl', 'BPGM+ctrl', 'C3orf72+ctrl', 'FOXL2+ctrl', 'CNN1+ctrl', 'ctrl+CDKN1A', 'ctrl+C19orf26', 'ARID1A+ctrl', 'ctrl+COL2A1', 'BCL2L11+ctrl', 'OSR2+ctrl', 'SET+ctrl', 'ctrl+SPI1', 'ctrl+STIL', 'CEBPB+ctrl', 'ctrl+PRTG', 'CDKN1B+ctrl', 'JUN+ctrl', 'PTPN13+ctrl', 'ctrl+MAPK1', 'PTPN9+ctrl', 'ctrl+SNAI1', 'ctrl+CEBPB', 'PRDM1+ctrl', 'ctrl+PTPN9', 'ctrl+OSR2', 'ctrl+FOXL2', 'SPI1+ctrl', 'SNAI1+ctrl', 'EGR1+ctrl', 'STIL+ctrl', 'CDKN1C+ctrl', 'ctrl+CDKN1B'], 'train': ['KLF1+MAP2K6', 'ctrl', 'CEBPE+RUNX1T1', 'ctrl+CEBPE', 'TGFBR2+ETS2', 'SGK1+TBX3', 'ctrl+ELMSAN1', 'ctrl+FOXA1', 'FOXA3+FOXA1', 'ETS2+IGDCC3', 'GLB1L2+ctrl', 'KLF1+ctrl', 'MAP2K6+IKZF3', '

In [17]:
import os
import csv
from collections import defaultdict

def _load_full_graph_from_file(file_path, gene_name_to_idx, device):
    """
    Internal helper to load one full graph from a CSV file.
    
    Args:
        file_path (str): Path to the CSV file.
        gene_name_to_idx (dict): {'gene_name': gene_idx} mapping.
        device (torch.device): Device to send tensors to.
        
    Returns:
        A torch_geometric.data.Data object (edge_index, edge_weight) or None.
    """
    if not os.path.exists(file_path):
        # print(f"  Info: File not found, skipping: {file_path}")
        return None
    
    edge_list = []
    weight_list = []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            header = next(reader) # Skip header
            
            for row in reader:
                if len(row) < 3: continue
                source_name, target_name, weight_str = row[0], row[1], row[2]
                
                u_idx = gene_name_to_idx.get(source_name)
                v_idx = gene_name_to_idx.get(target_name)
                
                # Skip edge if either gene is not in the map
                if u_idx is None or v_idx is None:
                    continue 
                
                try:
                    weight = float(weight_str)
                    edge_list.append([u_idx, v_idx])
                    weight_list.append(weight)
                except ValueError:
                    continue # Skip if weight is not a valid float

        if not edge_list:
            print(f"  Warning: No valid edges found in {file_path}")
            return None
            
        # Create tensors
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous().to(device)
        edge_weight = torch.tensor(weight_list, dtype=torch.float).to(device)
        
        graph_data = Data(edge_index=edge_index, edge_weight=edge_weight)
        return graph_data

    except Exception as e:
        print(f"  Error reading file {file_path}: {e}")
        return None

# --------
def standalone_load_ground_truth_graphs(
    node_map_gene,
    node_map_pert,
    device,
    base_path="/kaggle/input/coexpressiongraphs/Downloads/coexpression_graphs", 
    filename="_42_5045_0.4_20_co_expression_network.csv"
):
    """
    Loads full ground truth co-expression graphs from CSV files.

    This function loads the *entire* graph from each file, not just
    edges matching a predefined index.
    
    It returns:
    1.  gt_graph_store (dict): {pert_name: Data(edge_index, edge_weight)}
    2.  gt_jaccard_scores (dict): {pert_name: structural_jaccard_vs_ctrl}
    
    Args:
        node_map_gene (dict): A dict {'gene_name': gene_idx}.
        node_map_pert (dict): A dict {'pert_name': pert_idx}.
        device (torch.device or str): The device to create new tensors on.
        base_path (str): Base directory for GT graphs.
        filename (str): Name of the edges file.
    """
    print("Attempting to load full ground truth co-expression graphs...")
    
    gt_graph_store = {}
    gt_jaccard_scores = {}
    gene_name_to_idx = node_map_gene

    # --- 1. Load Control Graph First (as baseline) ---
    ctrl_file_path = "/kaggle/input/coexpressiongraphs/Downloads/coexpression_graphs/control/_42_5045_0.4_20_co_expression_network.csv"
    ctrl_graph = _load_full_graph_from_file(ctrl_file_path, gene_name_to_idx, device)
    
    if ctrl_graph is None:
        print(f"  FATAL ERROR: Control graph not found at {ctrl_file_path}. Aborting.")
        return {}, {}
    
    gt_graph_store['ctrl'] = ctrl_graph
    # Create a canonical set of edges (u, v) where u <= v
    ctrl_edge_set = set(
        tuple(sorted(edge)) for edge in ctrl_graph.edge_index.t().tolist()
    )
    print(f"  Loaded 'ctrl' graph with {len(ctrl_edge_set)} unique edges.")

    # --- 2. Load all other perturbation graphs ---
    pert_names = list(node_map_pert.keys())
    print(base_path)
    for pert_name in pert_names:
        if pert_name == 'ctrl': # Already loaded
            continue
        
        # file_path2 = os.path.join(base_path, f"pert_{pert_name}+ctrl", filename)   
        file_path = f"{base_path}/pert_{pert_name}+ctrl/{filename}"
        # print(file_path)
        
        pert_graph = _load_full_graph_from_file(file_path, gene_name_to_idx, device)
        
        if pert_graph is not None:
            gt_graph_store[pert_name] = pert_graph
            
            # --- 3. Compute Structural Jaccard Similarity ---
            pert_edge_set = set(
                tuple(sorted(edge)) for edge in pert_graph.edge_index.t().tolist()
            )
            
            intersection = len(ctrl_edge_set.intersection(pert_edge_set))
            union = len(ctrl_edge_set.union(pert_edge_set))
            
            jaccard_sim = intersection / union if union > 0 else 0.0
            gt_jaccard_scores[pert_name] = jaccard_sim
            
            print(f"  Loaded '{pert_name}' graph ({len(pert_edge_set)} edges). "
                  f"Jaccard vs. Ctrl: {jaccard_sim:.4f}")

    print(f"Successfully loaded {len(gt_graph_store)} total ground truth graphs.")
    return gt_graph_store, gt_jaccard_scores

def aggregate_predicted_graphs(model, dataloader, idx_to_pert_name, device):
    """
    Analyzes all cell graphs and returns the *averaged* graph 
    weights for each perturbation.
    
    Returns:
        A dict: { pert_name: {'mean_weights': 1D_Tensor, 'count': N} }
    """
    print("Starting predicted graph aggregation...")
    
    try:
        num_edges = model.G_coexpress_weight.shape[0]
    except AttributeError:
        print("Error: `model.G_coexpress_weight` not found. Aborting.")
        return {}
        
    # Temporary store for *all* weights
    pred_weights_store = defaultdict(list)

    # --- 1. Iterate Through Dataloader to Collect Weights ---
    for step, batch in enumerate(dataloader):
        with torch.no_grad():
            batch.to(device)
            pred, graph_data = model.compute_graphs(batch)
            
            predicted_graphs_list = graph_data.get("graphs")
            if predicted_graphs_list is None: continue
            
            pert_indices = np.array(batch.pert)
            if len(predicted_graphs_list) != pert_indices.shape[0]: continue

            for i in range(len(predicted_graphs_list)):
                pert_idx = pert_indices[i].item()
                pert_name = idx_to_pert_name.get(pert_idx, f"Unknown_idx_{pert_idx}")
                
                pred_weights_i = predicted_graphs_list[i].edge_weight.detach()
                
                if pred_weights_i.shape[0] == num_edges:
                    pred_weights_store[pert_name].append(pred_weights_i)
    
    print(f"  Scan complete. Found predicted graphs for {len(pred_weights_store)} perturbations.")

    # --- 2. Aggregate by Averaging ---
    aggregated_pred_store = {}
    for pert_name, weights_list in pred_weights_store.items():
        if not weights_list:
            continue
            
        stacked_weights = torch.stack(weights_list, dim=0)
        mean_pred_weights = torch.mean(stacked_weights, dim=0)
        count = stacked_weights.shape[0]
        
        aggregated_pred_store[pert_name] = {
            'mean_weights': mean_pred_weights,
            'count': count
        }
        
    print(f"  Averaged weights for {len(aggregated_pred_store)} perturbations.")
    return aggregated_pred_store

# --- Helpers for Function 3 ---

def _calculate_pearson(x, y):
    """Calculates Pearson correlation for two 1D tensors."""
    vx = x - torch.mean(x)
    vy = y - torch.mean(y)
    corr = torch.sum(vx * vy) / (torch.sqrt(torch.sum(vx ** 2)) * torch.sqrt(torch.sum(vy ** 2)))
    return corr.item()

def _calculate_weighted_jaccard(pred_weights, gt_weights):
    """Calculates the Tanimoto similarity for weighted, non-negative vectors."""
    pred_abs = torch.abs(pred_weights)
    gt_abs = torch.abs(gt_weights)
    min_sum = torch.sum(torch.minimum(pred_abs, gt_abs))
    max_sum = torch.sum(torch.maximum(pred_abs, gt_abs))
    if max_sum == 0:
        return 1.0 if min_sum == 0 else 0.0
    return (min_sum / max_sum).item()

def _project_gt_graph(gt_data, control_edge_index, device):
    """
    Projects the weights of a full GT graph onto the model's fixed control_edge_index.
    """
    num_control_edges = control_edge_index.shape[1]
    gt_weights_on_control = torch.zeros(num_control_edges, device=device)
    
    gt_map = {}
    gt_edges = gt_data.edge_index.cpu().t().tolist()
    gt_weights = gt_data.edge_weight.cpu().tolist()
    
    # Build a lookup map for the GT graph's weights
    for (u, v), w in zip(gt_edges, gt_weights):
        gt_map[(u, v)] = w
        gt_map[(v, u)] = w # Assume undirected
        
    # Iterate over the *model's* fixed edge structure
    control_edges_list = control_edge_index.cpu().t().tolist()
    for i, (u, v) in enumerate(control_edges_list):
        # Pull the weight from the GT map, default to 0.0
        weight = gt_map.get((u, v), 0.0) 
        gt_weights_on_control[i] = weight
        
    return gt_weights_on_control.to(device)

# --- NEW COMPARISON FUNCTION 3 ---
import re
def compare_aggregated_graphs(
    aggregated_pred_store, 
    gt_graph_store, 
    gt_jaccard_scores, 
    model, 
    device
):
    """
    Compares the averaged predicted graphs against the full ground truth graphs.
    """
    print("\n--- 📊 Aggregated Graph Comparison Report ---")
    final_report = {}
    
    try:
        control_edge_index = model.G_coexpress.to(device)
        control_weights = model.G_coexpress_weight.to(device).detach()
    except AttributeError:
        print("Error: `model.G_coexpress` or `model.G_coexpress_weight` not found. Aborting.")
        return {}
        
    if 'ctrl' not in gt_graph_store:
        print("Error: 'ctrl' graph not found in Ground Truth store. Aborting.")
        return {}
        
    # Project the GT control graph onto the model's edge index once
    gt_ctrl_weights_on_control = _project_gt_graph(
        gt_graph_store['ctrl'], control_edge_index, device
    )

    for pert_name, pred_data in aggregated_pred_store.items():
        
        mean_pred_weights = pred_data['mean_weights']
        count = pred_data['count']
        # match = re.search(r'([A-Z0-9]+)', pert_name
        pair = re.findall(r"Unknown_idx_([^ ()]+)", pert_name)
        if(len(pair)==0):
            continue
        parts = [g for g in pair[0].split("+") if g != "ctrl"]

        if(len(parts)==0):
            continue
        true_name = parts[0]
        # print(match.group(1))  # Output: RUNX1T1
        # --- Get Corresponding Ground Truth Graph ---
        gt_full_graph = gt_graph_store.get(true_name)
        
        if gt_full_graph is None:
            print(f"\n**Perturbation: {pert_name} (n={count})**")
            print(true_name)
            print(f"  (No Ground Truth graph loaded for this perturbation)")
            continue

        # --- Project GT Graph onto Model's Edge Index ---
        gt_weights_on_control = _project_gt_graph(
            gt_full_graph, control_edge_index, device
        )
        
        # --- Compute Metrics vs. GT ---
        gt_pearson_corr = _calculate_pearson(mean_pred_weights, gt_weights_on_control)
        gt_cos_sim = F.cosine_similarity(
            mean_pred_weights.unsqueeze(0), 
            gt_weights_on_control.unsqueeze(0)
        ).item()
        gt_weighted_jaccard = _calculate_weighted_jaccard(
            mean_pred_weights, gt_weights_on_control
        )
        
        # --- Compute Metrics vs. Control Graphs ---
        diff_vs_learned_ctrl = torch.mean(torch.abs(mean_pred_weights - control_weights)).item()
        diff_vs_gt_ctrl = torch.mean(torch.abs(mean_pred_weights - gt_ctrl_weights_on_control)).item()

        # --- Store and Print Report ---
        structural_jaccard = gt_jaccard_scores.get(pert_name, np.nan)
        final_report[pert_name] = {
            'count': count,
            'gt_pearson': gt_pearson_corr,
            'gt_cosine_sim': gt_cos_sim,
            'gt_weighted_jaccard': gt_weighted_jaccard,
            'gt_structural_jaccard': structural_jaccard,
            'mean_abs_diff_vs_learned_ctrl': diff_vs_learned_ctrl,
            'mean_abs_diff_vs_gt_ctrl': diff_vs_gt_ctrl
        }
        
        print(f"\n**Perturbation: {pert_name} (n={count})**")
        print(f"  --- vs. Ground Truth '{pert_name}' Graph ---")
        print(f"    Pearson Correlation:      {gt_pearson_corr:.4f}")
        print(f"    Cosine Similarity:        {gt_cos_sim:.4f}")
        print(f"    Weighted Jaccard (Tanimoto): {gt_weighted_jaccard:.4f}")
        print(f"    Structural Jaccard vs. Ctrl: {structural_jaccard:.4f}")
        print(f"  --- vs. Control Graphs ---")
        print(f"    Mean Abs. Diff (vs. *Learned* Ctrl): {diff_vs_learned_ctrl:.5f}")
        print(f"    Mean Abs. Diff (vs. *GT* Ctrl):      {diff_vs_gt_ctrl:.5f}")

    print("-----------------------------------")
    return final_report

In [18]:
OUTPUT_PATH = "model_output"
# gears_mech  = GEARS_PRETRAIN(pert_data=ctrl_pertdata, device='cuda', weight_bias_track=False, proj_name='GEARS', exp_name='GEARS')
gears_mech  = GEARS_PRETRAIN(pert_data=pert_data, device='cuda', weight_bias_track=False, proj_name='GEARS', exp_name='GEARS')

gears_mech.model_initialize(
    hidden_size=64, num_go_gnn_layers=1,
    num_gene_gnn_layers=1, decoder_hidden_size=16,
    num_similar_genes_go_graph=20, num_similar_genes_co_express_graph=20,
    coexpress_threshold=0.4, uncertainty=False, uncertainty_reg=1,
    direction_lambda=0.1, G_go=None, G_go_weight=None,
    G_coexpress=None, G_coexpress_weight=None, no_perturb=False,
    gears_model=True, model_no=1,
    num_hops=3,num_add=6)



Graph modification in progress. Original edge count: 6359


Processing nodes:   0%|          | 0/13374 [00:00<?, ?it/s]

Downloading...


Added 1443 new zero-weight edges.
Tensors updated. New edge count: 7802


100%|██████████| 60.7M/60.7M [00:02<00:00, 28.7MiB/s]
Extracting tar file...
Done!


Downprojected Rank 1560
Expected Dimensions of activations through encoder: [64, 128, 128]


In [19]:
##using mmd
# gears_mech.pretrain_ctrl(epochs=20, lr=1e-3, weight_decay=5e-4)
# In your training script, before calling gears_mech.train()
torch.autograd.set_detect_anomaly(True)

# gears_mech.pretrain_ctrl(epochs=20, lr=1e-3, weight_decay=5e-4)
deeperanalysis,nondropoutanalysis = gears_mech.train(epochs=4,lr=1e-3,weight_decay=5e-4,use_mmd=True)
gears_mech.save_model(OUTPUT_PATH)
## writing out control_adata expression

Start Training...


Training Phase with perturbing graph .....
Length of train_loader: 1503
Length of val_loader 96


Epoch 1 Step 1 Train Loss: 0.1112
Epoch 1 Step 51 Train Loss: 0.1093
Epoch 1 Step 101 Train Loss: 0.1213
Epoch 1 Step 151 Train Loss: 0.1044
Epoch 1 Step 201 Train Loss: 0.1051
Epoch 1 Step 251 Train Loss: 0.1231
Epoch 1 Step 301 Train Loss: 0.0963
Epoch 1 Step 351 Train Loss: 0.1026
Epoch 1 Step 401 Train Loss: 0.1169
Epoch 1 Step 451 Train Loss: 0.1003
Epoch 1 Step 501 Train Loss: 0.0962
Epoch 1 Step 551 Train Loss: 0.0939
Epoch 1 Step 601 Train Loss: 0.1004
Epoch 1 Step 651 Train Loss: 0.1233
Epoch 1 Step 701 Train Loss: 0.0969
Epoch 1 Step 751 Train Loss: 0.1140
Epoch 1 Step 801 Train Loss: 0.1022
Epoch 1 Step 851 Train Loss: 0.1210
Epoch 1 Step 901 Train Loss: 0.1077
Epoch 1 Step 951 Train Loss: 0.1219
Epoch 1 Step 1001 Train Loss: 0.1072
Epoch 1 Step 1051 Train Loss: 0.1000
Epoch 1 Step 1101 Train Loss: 0.1082
Epoch 1 Step 1151 Train Loss: 0.1109
Epoch 1 Step 1201 Train Loss: 0.1020
Epoch 1 Step 1251 Train Loss: 0.1068
Epoch 1 Step 1301 Train Loss: 0.1105
Epoch 1 Step 1351 Train 

New Best model has val_mse: 0.0034232182176007577


Epoch 2 Step 1 Train Loss: 0.0974
Epoch 2 Step 51 Train Loss: 0.1099
Epoch 2 Step 101 Train Loss: 0.1064
Epoch 2 Step 151 Train Loss: 0.1116
Epoch 2 Step 201 Train Loss: 0.1115
Epoch 2 Step 251 Train Loss: 0.1061
Epoch 2 Step 301 Train Loss: 0.1092
Epoch 2 Step 351 Train Loss: 0.1209
Epoch 2 Step 401 Train Loss: 0.1132
Epoch 2 Step 451 Train Loss: 0.1059
Epoch 2 Step 501 Train Loss: 0.1091
Epoch 2 Step 551 Train Loss: 0.1053
Epoch 2 Step 601 Train Loss: 0.1236
Epoch 2 Step 651 Train Loss: 0.0974
Epoch 2 Step 701 Train Loss: 0.1041
Epoch 2 Step 751 Train Loss: 0.1027
Epoch 2 Step 801 Train Loss: 0.0995
Epoch 2 Step 851 Train Loss: 0.1170
Epoch 2 Step 901 Train Loss: 0.1193
Epoch 2 Step 951 Train Loss: 0.1113
Epoch 2 Step 1001 Train Loss: 0.1179
Epoch 2 Step 1051 Train Loss: 0.1149
Epoch 2 Step 1101 Train Loss: 0.1134
Epoch 2 Step 1151 Train Loss: 0.1163
Epoch 2 Step 1201 Train Loss: 0.1074
Epoch 2 Step 1251 Train Loss: 0.1192
Epoch 2 Step 1301 Train Loss: 0.1021
Epoch 2 Step 1351 Train 

{'frac_correct_direction_all': 0.453465241398839, 'frac_correct_direction_20': 0.6214285714285712, 'frac_correct_direction_50': 0.6321428571428569, 'frac_correct_direction_100': 0.6139285714285715, 'frac_correct_direction_200': 0.5885714285714286, 'frac_correct_direction_20_nonzero': 0.8934089696555465, 'frac_in_range': 0.9896301368050593, 'frac_in_range_45_55': 0.10791822490441208, 'frac_in_range_40_60': 0.22504654358369527, 'frac_in_range_25_75': 0.57741125841492, 'mean_sigma': 0.702414044393943, 'std_sigma': 0.4272825915652972, 'frac_sigma_below_1': 0.8142007963260046, 'frac_sigma_below_2': 0.9743368956894499, 'pearson_delta': 0.4337940572627953, 'pearson_delta_de': 0.42249374011797564, 'fold_change_gap_all': 0.46360109906111446, 'fold_change_gap_upreg_3': 22.78006599393002, 'fold_change_gap_upreg_10': 125.77368541197343, 'pearson_delta_top200_hvg': 0.4856793758060251, 'pearson_top200_hvg': 0.9691813343337604, 'mse_top200_hvg': 0.05245473947642105, 'pearson_delta_top20_de': 0.422493

In [20]:
gt_graph_store, gt_jaccard_scores = standalone_load_ground_truth_graphs(
    gears_mech.node_map,
    gears_mech.node_map_pert,
    gears_mech.device,
    base_path="/kaggle/input/coexpressiongraphs/Downloads/coexpression_graphs", 
    filename="_42_5045_0.4_20_co_expression_network.csv"
)

Attempting to load full ground truth co-expression graphs...
  Loaded 'ctrl' graph with 5229 unique edges.
/kaggle/input/coexpressiongraphs/Downloads/coexpression_graphs
  Loaded 'AHR' graph (7196 edges). Jaccard vs. Ctrl: 0.2441
  Loaded 'ARID1A' graph (9563 edges). Jaccard vs. Ctrl: 0.1579
  Loaded 'ARRDC3' graph (5439 edges). Jaccard vs. Ctrl: 0.2635
  Loaded 'ATL1' graph (6159 edges). Jaccard vs. Ctrl: 0.2264
  Loaded 'BAK1' graph (4799 edges). Jaccard vs. Ctrl: 0.3042
  Loaded 'BCL2L11' graph (5605 edges). Jaccard vs. Ctrl: 0.2694
  Loaded 'BCORL1' graph (5815 edges). Jaccard vs. Ctrl: 0.2632
  Loaded 'BPGM' graph (6284 edges). Jaccard vs. Ctrl: 0.2377
  Loaded 'C19orf26' graph (5432 edges). Jaccard vs. Ctrl: 0.2780
  Loaded 'C3orf72' graph (7856 edges). Jaccard vs. Ctrl: 0.1874
  Loaded 'CBFA2T3' graph (7040 edges). Jaccard vs. Ctrl: 0.2139
  Loaded 'CBL' graph (5758 edges). Jaccard vs. Ctrl: 0.2770
  Loaded 'CDKN1A' graph (10677 edges). Jaccard vs. Ctrl: 0.1439
  Loaded 'CDKN1B'

In [21]:
aggregated_pred_store = aggregate_predicted_graphs(gears_mech.model, gears_mech.dataloader["train_loader"], gears_mech.node_map, gears_mech.device)

Starting predicted graph aggregation...
  Scan complete. Found predicted graphs for 129 perturbations.
  Averaged weights for 129 perturbations.


In [22]:
final_report = compare_aggregated_graphs(
    aggregated_pred_store, 
    gt_graph_store, 
    gt_jaccard_scores, 
    gears_mech.model, 
    gears_mech.device
)


--- 📊 Aggregated Graph Comparison Report ---

**Perturbation: Unknown_idx_KLF1+ctrl (n=997)**
  --- vs. Ground Truth 'Unknown_idx_KLF1+ctrl' Graph ---
    Pearson Correlation:      0.0149
    Cosine Similarity:        0.6003
    Weighted Jaccard (Tanimoto): 0.2986
    Structural Jaccard vs. Ctrl: nan
  --- vs. Control Graphs ---
    Mean Abs. Diff (vs. *Learned* Ctrl): 0.49154
    Mean Abs. Diff (vs. *GT* Ctrl):      0.66192

**Perturbation: Unknown_idx_UBASH3A+ctrl (n=371)**
  --- vs. Ground Truth 'Unknown_idx_UBASH3A+ctrl' Graph ---
    Pearson Correlation:      0.0132
    Cosine Similarity:        0.5559
    Weighted Jaccard (Tanimoto): 0.2560
    Structural Jaccard vs. Ctrl: nan
  --- vs. Control Graphs ---
    Mean Abs. Diff (vs. *Learned* Ctrl): 0.49154
    Mean Abs. Diff (vs. *GT* Ctrl):      0.66192

**Perturbation: Unknown_idx_ETS2+ctrl (n=375)**
  --- vs. Ground Truth 'Unknown_idx_ETS2+ctrl' Graph ---
    Pearson Correlation:      0.0174
    Cosine Similarity:        0.5379


In [23]:
# gears_mech.pretrain_ctrl(epochs=20, lr=1e-3, weight_decay=5e-4)
# In your training script, before calling gears_mech.train()
OUTPUT_PATH = "model_output"
# gears_mech  = GEARS_PRETRAIN(pert_data=ctrl_pertdata, device='cuda', weight_bias_track=False, proj_name='GEARS', exp_name='GEARS')
gears_mech  = GEARS_PRETRAIN(pert_data=pert_data, device='cuda', weight_bias_track=False, proj_name='GEARS', exp_name='GEARS')

gears_mech.model_initialize(
    hidden_size=64, num_go_gnn_layers=1,
    num_gene_gnn_layers=1, decoder_hidden_size=16,
    num_similar_genes_go_graph=20, num_similar_genes_co_express_graph=20,
    coexpress_threshold=0.4, uncertainty=False, uncertainty_reg=1,
    direction_lambda=0.1, G_go=None, G_go_weight=None,
    G_coexpress=None, G_coexpress_weight=None, no_perturb=False,
    gears_model=True, model_no=1,
    num_hops=3,num_add=20)


torch.autograd.set_detect_anomaly(True)

# gears_mech.pretrain_ctrl(epochs=20, lr=1e-3, weight_decay=5e-4)
deeperanalysis,nondropoutanalysis = gears_mech.train(epochs=4,lr=1e-3,weight_decay=5e-4)
gears_mech.save_model(OUTPUT_PATH)
## writing out control_adata expression

Graph modification in progress. Original edge count: 6359


Processing nodes:   0%|          | 0/13374 [00:00<?, ?it/s]

Found local copy...


Added 3016 new zero-weight edges.
Tensors updated. New edge count: 9375


Start Training...


Downprojected Rank 1875
Expected Dimensions of activations through encoder: [64, 128, 128]
Training Phase with perturbing graph .....
Length of train_loader: 1503
Length of val_loader 96


Epoch 1 Step 1 Train Loss: 0.5214
Epoch 1 Step 51 Train Loss: 0.3714
Epoch 1 Step 101 Train Loss: 0.3084
Epoch 1 Step 151 Train Loss: 0.3328
Epoch 1 Step 201 Train Loss: 0.3775
Epoch 1 Step 251 Train Loss: 0.4411
Epoch 1 Step 301 Train Loss: 0.3968
Epoch 1 Step 351 Train Loss: 0.3972
Epoch 1 Step 401 Train Loss: 0.4189
Epoch 1 Step 451 Train Loss: 0.3738
Epoch 1 Step 501 Train Loss: 0.3458
Epoch 1 Step 551 Train Loss: 0.3709
Epoch 1 Step 601 Train Loss: 0.3528
Epoch 1 Step 651 Train Loss: 0.3851
Epoch 1 Step 701 Train Loss: 0.4087
Epoch 1 Step 751 Train Loss: 0.3714
Epoch 1 Step 801 Train Loss: 0.3603
Epoch 1 Step 851 Train Loss: 0.3798
Epoch 1 Step 901 Train Loss: 0.4406
Epoch 1 Step 951 Train Loss: 0.4204
Epoch 1 Step 1001 Train Loss: 0.3964
Epoch 1 Step 1051 Train Loss: 0.4451
Epoch 1 Step 1101 Train Loss: 0.4145
Epoch 1 Step 1151 Train Loss: 0.3621
Epoch 1 Step 1201 Train Loss: 0.4325
Epoch 1 Step 1251 Train Loss: 0.3716
Epoch 1 Step 1301 Train Loss: 0.3801
Epoch 1 Step 1351 Train 

New Best model has val_mse: 0.009106602054089308


Epoch 2 Step 1 Train Loss: 0.3674
Epoch 2 Step 51 Train Loss: 0.3840
Epoch 2 Step 101 Train Loss: 0.3930
Epoch 2 Step 151 Train Loss: 0.4156
Epoch 2 Step 201 Train Loss: 0.4798
Epoch 2 Step 251 Train Loss: 0.4185
Epoch 2 Step 301 Train Loss: 0.4361
Epoch 2 Step 351 Train Loss: 0.4098
Epoch 2 Step 401 Train Loss: 0.4028
Epoch 2 Step 451 Train Loss: 0.3838
Epoch 2 Step 501 Train Loss: 0.3912
Epoch 2 Step 551 Train Loss: 0.4457
Epoch 2 Step 601 Train Loss: 0.4323
Epoch 2 Step 651 Train Loss: 0.3998
Epoch 2 Step 701 Train Loss: 0.4265
Epoch 2 Step 751 Train Loss: 0.3729
Epoch 2 Step 801 Train Loss: 0.4071
Epoch 2 Step 851 Train Loss: 0.4315
Epoch 2 Step 901 Train Loss: 0.4059
Epoch 2 Step 951 Train Loss: 0.4245
Epoch 2 Step 1001 Train Loss: 0.4126
Epoch 2 Step 1051 Train Loss: 0.3678
Epoch 2 Step 1101 Train Loss: 0.4340
Epoch 2 Step 1151 Train Loss: 0.4353
Epoch 2 Step 1201 Train Loss: 0.4037
Epoch 2 Step 1251 Train Loss: 0.3990
Epoch 2 Step 1301 Train Loss: 0.4216
Epoch 2 Step 1351 Train 

{'frac_correct_direction_all': 0.4239664448534617, 'frac_correct_direction_20': 0.5517857142857142, 'frac_correct_direction_50': 0.5271428571428571, 'frac_correct_direction_100': 0.4896428571428572, 'frac_correct_direction_200': 0.4507142857142857, 'frac_correct_direction_20_nonzero': 0.8994700383267603, 'frac_in_range': 0.9934200603318251, 'frac_in_range_45_55': 0.09241302329091085, 'frac_in_range_40_60': 0.2045916490233589, 'frac_in_range_25_75': 0.5722185156373357, 'mean_sigma': 0.9081223197281361, 'std_sigma': 0.942911408125208, 'frac_sigma_below_1': 0.7531382106719986, 'frac_sigma_below_2': 0.9615039436827362, 'pearson_delta': 0.3192703361356897, 'pearson_delta_de': 0.44481012018929633, 'fold_change_gap_all': 0.43667623826435636, 'fold_change_gap_upreg_3': 21.315793880196505, 'fold_change_gap_upreg_10': 121.49715896086259, 'pearson_delta_top200_hvg': 0.43245958084506647, 'pearson_top200_hvg': 0.9591254485504968, 'mse_top200_hvg': 0.07908106329185623, 'pearson_delta_top20_de': 0.44

In [24]:
gt_graph_store, gt_jaccard_scores = standalone_load_ground_truth_graphs(
    gears_mech.node_map,
    gears_mech.node_map_pert,
    gears_mech.device,
    base_path="/kaggle/input/coexpressiongraphs/Downloads/coexpression_graphs", 
    filename="_42_5045_0.4_20_co_expression_network.csv"
)

Attempting to load full ground truth co-expression graphs...
  Loaded 'ctrl' graph with 5229 unique edges.
/kaggle/input/coexpressiongraphs/Downloads/coexpression_graphs
  Loaded 'AHR' graph (7196 edges). Jaccard vs. Ctrl: 0.2441
  Loaded 'ARID1A' graph (9563 edges). Jaccard vs. Ctrl: 0.1579
  Loaded 'ARRDC3' graph (5439 edges). Jaccard vs. Ctrl: 0.2635
  Loaded 'ATL1' graph (6159 edges). Jaccard vs. Ctrl: 0.2264
  Loaded 'BAK1' graph (4799 edges). Jaccard vs. Ctrl: 0.3042
  Loaded 'BCL2L11' graph (5605 edges). Jaccard vs. Ctrl: 0.2694
  Loaded 'BCORL1' graph (5815 edges). Jaccard vs. Ctrl: 0.2632
  Loaded 'BPGM' graph (6284 edges). Jaccard vs. Ctrl: 0.2377
  Loaded 'C19orf26' graph (5432 edges). Jaccard vs. Ctrl: 0.2780
  Loaded 'C3orf72' graph (7856 edges). Jaccard vs. Ctrl: 0.1874
  Loaded 'CBFA2T3' graph (7040 edges). Jaccard vs. Ctrl: 0.2139
  Loaded 'CBL' graph (5758 edges). Jaccard vs. Ctrl: 0.2770
  Loaded 'CDKN1A' graph (10677 edges). Jaccard vs. Ctrl: 0.1439
  Loaded 'CDKN1B'

In [25]:
aggregated_pred_store = aggregate_predicted_graphs(gears_mech.model, gears_mech.dataloader["train_loader"], gears_mech.node_map, gears_mech.device)

Starting predicted graph aggregation...
  Scan complete. Found predicted graphs for 129 perturbations.
  Averaged weights for 129 perturbations.


In [26]:
final_report = compare_aggregated_graphs(
    aggregated_pred_store, 
    gt_graph_store, 
    gt_jaccard_scores, 
    gears_mech.model, 
    gears_mech.device
)


--- 📊 Aggregated Graph Comparison Report ---

**Perturbation: Unknown_idx_ZBTB10+DLX2 (n=74)**
  --- vs. Ground Truth 'Unknown_idx_ZBTB10+DLX2' Graph ---
    Pearson Correlation:      0.0091
    Cosine Similarity:        0.4911
    Weighted Jaccard (Tanimoto): 0.2039
    Structural Jaccard vs. Ctrl: nan
  --- vs. Control Graphs ---
    Mean Abs. Diff (vs. *Learned* Ctrl): 0.56361
    Mean Abs. Diff (vs. *GT* Ctrl):      0.69580

**Perturbation: Unknown_idx_FOXF1+ctrl (n=448)**
  --- vs. Ground Truth 'Unknown_idx_FOXF1+ctrl' Graph ---
    Pearson Correlation:      0.0133
    Cosine Similarity:        0.4989
    Weighted Jaccard (Tanimoto): 0.2139
    Structural Jaccard vs. Ctrl: nan
  --- vs. Control Graphs ---
    Mean Abs. Diff (vs. *Learned* Ctrl): 0.56361
    Mean Abs. Diff (vs. *GT* Ctrl):      0.69580

**Perturbation: Unknown_idx_KLF1+BAK1 (n=323)**
  --- vs. Ground Truth 'Unknown_idx_KLF1+BAK1' Graph ---
    Pearson Correlation:      0.0244
    Cosine Similarity:        0.5457
 